# Nouvelle section Partie Clients / Equipements / Etiquettes

In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling
!pip install rapidfuzz

In [ ]:
# definitions d'une fonction
from rapidfuzz import fuzz
import pandas as pd

def sont_doublons(a, b, seuil=90):
    if pd.isna(a) or pd.isna(b):
        return False
    return fuzz.token_sort_ratio(a.lower(), b.lower()) >= seuil

In [ ]:
import requestsa
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import unicodedata
import re
import numpy as np
import pandas as pd

In [ ]:
def nettoyer_html(texte):
    if pd.isna(texte):
        return texte

    soup = BeautifulSoup(texte, "html.parser")

    # transforme les <br> en retours à la ligne
    for br in soup.find_all("br"):
        br.replace_with("\n")

    texte = soup.get_text(separator=" ", strip=True)

    return texte

In [ ]:
#lire le fichier csv CONTACT ODOO
df_Eq = pd.read_excel('/content/Contact (res.partner) (Equipements).xlsx');
df_Et = pd.read_excel('/content/Contact (res.partner) (Etiquettes).xlsx');
#Afficher tout le df
pd.set_option('display.max_rows', None)

In [ ]:
# Fonction de nettoyage des ","
def clean_value(x):
    if pd.isna(x):
        return np.nan

    x = str(x)

    for _ in range(3):  # début
        if x.startswith(","):
            x = x[1:].lstrip()

    for _ in range(3):  # fin
        if x.endswith(","):
            x = x[:-1].rstrip()

    return np.nan if x == "" else x

In [ ]:
def normalize_name(name):
    name = str(name).strip().upper()
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = re.sub(r"[^A-Z0-9 ]", "", name)
    return name

def company_id_smart(name, max_len=5):
    if pd.isna(name):
        return np.nan

    name = normalize_name(name)
    words = name.split()

    if not words:
        return np.nan

    first_word = words[0]

    # Si déjà court (AXA, IBM, EDF, SCA…)
    if len(first_word) <= max_len:
        return first_word

    # Essayer un préfixe plus "distinctif"
    # (évite que SANOFI / SANTANDER → SANOF)
    prefix = first_word[:max_len]

    return prefix

In [ ]:
dico = {}         # dictionnaire final
Liste = []        # liste courante d'équipements
Nom = None        # clé courante

for x, y in zip(df_Eq['Nom affiché'], df_Eq['Equipements']):
    if pd.notna(x) and str(x).strip() != "":
        # On change de client : si un précédent existait, on l'enregistre
        if Nom is not None:
            dico[Nom] = Liste
        # Nouvelle clé + on réinitialise la liste
        Nom = str(x).strip()
        Liste = []

    # On ajoute l'équipement si présent
    if pd.notna(y) and str(y).strip() != "":
        Liste.append(str(y).strip())

# A la fin, ne pas oublier d'enregistrer le dernier client
if Nom is not None:
    dico[Nom] = Liste

print(dico)

In [ ]:
# on supprime les espaces dans la colonne
df_Eq['Nom affiché'] = df_Eq['Nom affiché'].str.strip()

# On mappe dico sur la colonne Nom affiché
mapped = df_Eq['Nom affiché'].map(dico)

# On écrit la liste uniquement là où Nom affiché n'est pas NaN
mask_client = df_Eq['Nom affiché'].notna() & (df_Eq['Nom affiché'].astype(str).str.strip() != '')
df_Eq.loc[mask_client, 'Equipements'] = mapped[mask_client]


# PARTIE SOCIETE

---



In [ ]:
Dico_ville = {

    # ===== KONE =====
    "KONE": "KONE",
    "kone": "KONE",
    "Koné": "KONE",
    "Kone": "KONE",
    "Village de Kone": "KONE",

    # ===== NOUMEA =====
    "NOUMEA": "NOUMEA",
    "98800": "NOUMEA",
    "Nouméa": "NOUMEA",
    "Noumea": "NOUMEA",
    "NOUMÉA": "NOUMEA",
    "98800 NOUMEA": "NOUMEA",
    "Nouméa Cédex": "NOUMEA",
    "Nouméa Cedex": "NOUMEA",
    "NOUMEA cedex": "NOUMEA",
    "NOUMEA CEDEX": "NOUMEA",
    "BP Q1 NOUMEA CEDEX": "NOUMEA",

    # ===== MONT DORE =====
    "Mont Dore": "MONT DORE",
    "Mont-Dore": "MONT DORE",
    "Mont-dore": "MONT DORE",
    "MONT-DORE": "MONT DORE",
    "MONT DORE": "MONT DORE",
    "Mont dore": "MONT DORE",

    # ===== PAITA =====
    "Paita": "PAITA",
    "Païta": "PAITA",
    "PAITA": "PAITA",
    "PAÏTA": "PAITA",

    # ===== DUMBEA =====
    "DUMBEA": "DUMBEA",
    "Dumbéa": "DUMBEA",
    "Dumbea": "DUMBEA",

    # ===== LA FOA =====
    "La foa": "LA FOA",
    "La Foa": "LA FOA",
    "LAFOA": "LA FOA",
    "LA FOA": "LA FOA",

    # ===== YATE =====
    "YATE": "YATE",
    "Yaté": "YATE",

    # ===== TOUHO =====
    "Touho": "TOUHO",
    "TOUHO": "TOUHO",

    # ===== POINDIMIE =====
    "POINDIMIE": "POINDIMIE",
    "Poindimié": "POINDIMIE",

    # ===== POUEMBOUT =====
    "Pouembout": "POUEMBOUT",
    "pouembout": "POUEMBOUT",
    "POUEMBOUT": "POUEMBOUT",

    # ===== KOUmac =====
    "KOUMAC": "KOUMAC",
    "Koumac": "KOUMAC",
    "koumac": "KOUMAC",

    # ===== BOURAIL =====
    "BOURAIL": "BOURAIL",
    "Bourail": "BOURAIL",

    # ===== AUTRES COMMUNES NC =====
    "OUEGOA": "OUEGOA",
    "OUVEA": "OUVEA",
    "UVEA": "UVEA",
    "LIFOU": "LIFOU",
    "Lifou": "LIFOU",
    "MARE": "MARE",
    "VAO": "VAO",
    "POUM": "POUM",
    "POYA": "POYA",
    "NEPOUI": "NEPOUI",
    "FARINO": "FARINO",
    "CANALA": "CANALA",
    "SARRAMEA": "SARRAMEA",
    "HOUAILOU": "HOUAILOU",
    "Houailou": "HOUAILOU",
    "HIENGHENE": "HIENGHENE",
    "PINDAI": "PINDAI",
    "EMIA": "EMIA",
    "KOUAOUA": "KOUAOUA",
    "OUACO": "OUACO",
    "VAVOUTO": "VAVOUTO",
    "Vavouto": "VAVOUTO",
    "POUEBO": "POUEBO",
    "Pouébo": "POUEBO",
    "PONHERIOUEN": "PONHERIOUEN",
    "TAMOA": "TAMOA",
    "YAHOUE": "YAHOUE",
    "VAO": "VAO",

    # ===== ILES =====
    "Ile des Pins": "ILE DES PINS",
    "ILE DES PINS": "ILE DES PINS",

    # ===== LIEUX COMPOSES =====
    "Koulnoue - Hienghene": "HIENGHENE",
    "Village": np.nan,
    "Nord": np.nan,

    # ===== HORS NC =====
    "PAPEETE": "PAPEETE",
    "Papeete": "PAPEETE",
    "WALLIS": "WALLIS",
    "Mata-Utu": "MATA-UTU",
    "Sigave": "SIGAVE",

    "PARIS": "PARIS",
    "DIJON": "DIJON",
    "METZ": "METZ",
    "MEAUX": "MEAUX",
    "ENTZHEIM": "ENTZHEIM",
    "SAINT CLOUD": "SAINT CLOUD",
    "Saint Cloud": "SAINT CLOUD",
    "MONTREUIL SOUS BOIS": "MONTREUIL SOUS BOIS",
    "VAULX EN VELIN": "VAULX EN VELIN",
    "Beaupreau en Mauges": "BEAUPREAU EN MAUGES",

    np.nan: np.nan
}

In [ ]:
Dico_pays = {
    'Nouvelle-Calédonie' : "NOUVELLE-CALEDONIE",
    'France' : "FRANCE",
    'Hong-Kong' : "HONG-KONG",
    'Polynésie française' : "POLYNESIE FRANCAISE",
    'Wallis et Futuna' : "WALLIS & FUTUNA",
    'Chine' : "CHINE",
    'Canada' : "CANADA"
}

In [ ]:
# dataframe des sociétés
df_société = df_Eq[df_Eq['Type de société'] == 'Société']

In [ ]:
#supprimer les lignes contenant des mails
df_société = df_société[~df_société['Nom affiché'].str.contains(r'@', na=False)]

In [ ]:
#Etape 1 : fonction pour regroupe les doublons
groupes = []
utilises = set()

for i, nom_i in df_société['Nom affiché'].items():
    if i in utilises:
        continue

    groupe = [i]
    for j, nom_j in df_société['Nom affiché'].items():
        if i != j and j not in utilises:
            if sont_doublons(nom_i, nom_j):
                groupe.append(j)

    groupes.append(groupe)
    utilises.update(groupe)

# Etape 2 : fusionner les doublons
def fusionner_lignes(df, indices):
    lignes = df.loc[indices]
    return lignes.ffill().bfill().iloc[0]

# Etape 3 : df fusionne
df_fusionne = pd.DataFrame(
    [fusionner_lignes(df_société, g) for g in groupes]
).reset_index(drop=True)

In [ ]:
# remplace les float nan par de vrai nan

df_fusionne["Ville"] = (
    df_fusionne["Ville"].str.strip()
    .replace(["nan", "None", "", '<NA>'], np.nan)
    .astype(object)
)
df_fusionne["Ville"] = df_fusionne["Ville"].replace({pd.NA: np.nan})

df_fusionne["Pays"] = (
    df_fusionne["Pays"].str.strip()
    .replace(["nan", "None", "", '<NA>'], np.nan)
    .astype(object)
)
df_fusionne["Pays"] = df_fusionne["Pays"].replace({pd.NA: np.nan})

In [ ]:
# normalise la donné à partir des dico
df_fusionne["Ville"] = df_fusionne["Ville"].map(Dico_ville)
df_fusionne["Pays"] = df_fusionne["Pays"].map(Dico_pays)
df_fusionne["Adresse de contact complète"] = df_fusionne["Adresse de contact complète"].apply(clean_value)

mask = df_fusionne["Ville"].str.upper().str.contains("NOUMEA", na=False)
df_fusionne.loc[mask, "Pays"] = "NOUVELLE-CALEDONIE"

df_fusionne["ID"] = df_fusionne["Nom affiché"].apply(company_id_smart)



In [ ]:
# Liste des villes à traiter

villes_nc = [
    "NOUMEA",
    "KONE",
    "PAITA",
    "DUMBEA",
    "MONT DORE",
    "LA FOA",
    "KOUMAC",
    "POUEMBOUT",
    "LIFOU",
    "BOURAIL",
    "POINDIMIE",
    "HIENGHENE",
    "HOUAILOU",
    "TOUHO",
    "OUEGOA",
    "YATE",
    "SARRAMEA",
    "KOUAOUA",
    "POUEBO",
    "PAPEETE",
    "PARIS",
    "FARINO",
    "ILE DES PINS",
    "CANALA",
    "NEPOUI",
    "PONHERIOUEN",
    "ENTZHEIM",
    "SIGAVE",
    "MARE",
    "EMIA",
    "PINDAI",
    "OUVEA",
    "MATA-UTU",
    "BEAUPREAU EN MAUGES",
    "MONTREUIL SOUS BOIS",
    "POUM",
    "POYA",
    "VAO",
    "TAMOA",
    "YAHOUE"
]


for ville in villes_nc:
    mask = df_fusionne["Ville"].str.contains(ville, na=False)
    df_fusionne.loc[mask, "Pays"] = "NOUVELLE-CALEDONIE"

mask = df_fusionne["Ville"].str.contains('PARIS', na=False)
df_fusionne.loc[mask, "Pays"] = "FRANCE"
mask = df_fusionne["Ville"].str.contains('DIJON', na=False)
df_fusionne.loc[mask, "Pays"] = "FRANCE"

In [ ]:
df_clients = df_fusionne[["ID",'Nom affiché','Courriel', 'Téléphone', 'Adresse de contact complète', 'Ville', 'Pays']]
df_clients.rename(columns={'Nom affiché': 'Entreprise', "ID": "ID", 'Courriel': 'Email', 'Téléphone': 'Téléphone', "Adresse de contact complète": "Adresse", 'Ville': 'Ville', 'Pays': 'Pays'}, inplace=True)

In [ ]:
# a run plusieurs fois
df_clients["Adresse"] = (
    df_clients["Adresse"]
    .astype(str)
    .str.rstrip(", ")   # enlève virgules ET espaces
)

# a run plusieurs fois
df_clients["Adresse"] = (
    df_clients["Adresse"]
    .astype(str)
    .str.rstrip(", ")   # enlève virgules ET espaces à la fin
)

# remplace les nan par de vrais nan
df_clients["Adresse"] = df_clients["Adresse"].replace({'nan': np.nan})

df_clients["Adresse"] = (
    df_clients["Adresse"]
    .str.replace(r",\s*Nouméa,\s*France", ", Nouméa", regex=True)
    .str.strip()
)

df_clients.loc[:, "Adresse"] = (
    df_clients["Adresse"]
    .str.replace(
        r"\s*Noum[eé]a,\s*France",
        " Nouméa",
        regex=True
    )
    .str.replace(r"\s+", " ", regex=True)  # nettoie espaces multiples
    .str.strip()
)

In [ ]:
#telecharger au format excel
output_file = "df_clients.xlsx"
df_clients.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)

### Partie Equipements

---



In [ ]:
rows = []

for _, row in df_fusionne.iterrows():
    nom = row["Nom affiché"]
    equip_list = row["Equipements"]

    # Cas 1 : liste vide ou NaN → une seule ligne avec NaN
    if not isinstance(equip_list, list) or len(equip_list) == 0:
        rows.append({
            "Nom affiché": nom,
            "Equipements": np.nan
        })

    # Cas 2 : liste non vide → une ligne par équipement
    else:
        for equip in equip_list:
            rows.append({
                "Nom affiché": nom,
                "Equipements": equip
            })

# Nouveau DataFrame avec seulement les 2 colonnes voulues
df_Equipements = pd.DataFrame(rows)

df_Equipements['Nom affiché'] = df_Equipements['Nom affiché'].str.strip()
df_Equipements['Equipements'] = df_Equipements['Equipements'].str.strip()

df_Equipements

In [ ]:
df_Equipements = df_Equipements.dropna(subset=['Equipements'])

In [ ]:
output_file = "df_Equipements.xlsx"
df_Equipements.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)

# Partie Etiquettes

---



In [ ]:
dico = {}         # dictionnaire final
Liste = []        # liste courante d'équipements
Nom = None        # clé courante

for x, y in zip(df_Et['Nom affiché'], df_Et['Étiquettes']):
    if pd.notna(x) and str(x).strip() != "":
        # On change de client : si un précédent existait, on l'enregistre
        if Nom is not None:
            dico[Nom] = Liste
        # Nouvelle clé + on réinitialise la liste
        Nom = str(x).strip()
        Liste = []

    # On ajoute l'équipement si présent
    if pd.notna(y) and str(y).strip() != "":
        Liste.append(str(y).strip())

# A la fin, ne pas oublier d'enregistrer le dernier client
if Nom is not None:
    dico[Nom] = Liste

print(dico)

In [ ]:
# on supprime les espaces dans la colonne
df_Et['Nom affiché'] = df_Et['Nom affiché'].str.strip()

# On mappe dico sur la colonne Nom affiché
mapped = df_Et['Nom affiché'].map(dico)

# On écrit la liste uniquement là où Nom affiché n'est pas NaN
mask_client = df_Et['Nom affiché'].notna() & (df_Et['Nom affiché'].astype(str).str.strip() != '')
df_Et.loc[mask_client, 'Étiquettes'] = mapped[mask_client]

In [ ]:
# dataframe des sociétés
df_Etiquettes = df_Et[df_Et['Type de société'] == 'Société']

#supprimer les lignes contenant des mails
df_Etiquettes = df_Etiquettes[~df_Etiquettes['Nom affiché'].str.contains(r'@', na=False)]

In [ ]:
#Etape 1 : fonction pour regroupe les doublons
groupes = []
utilises = set()

for i, nom_i in df_Etiquettes['Nom affiché'].items():
    if i in utilises:
        continue

    groupe = [i]
    for j, nom_j in df_Etiquettes['Nom affiché'].items():
        if i != j and j not in utilises:
            if sont_doublons(nom_i, nom_j):
                groupe.append(j)

    groupes.append(groupe)
    utilises.update(groupe)

# Etape 2 : fusionner les doublons
def fusionner_lignes(df, indices):
    lignes = df.loc[indices]
    return lignes.ffill().bfill().iloc[0]

# Etape 3 : df fusionne
df_Etiquettes_fusion = pd.DataFrame(
    [fusionner_lignes(df_Etiquettes, g) for g in groupes]
).reset_index(drop=True)

In [ ]:
rows = []

for _, row in df_Etiquettes_fusion.iterrows():
    nom = row["Nom affiché"]
    equip_list = row["Étiquettes"]

    # Cas 1 : liste vide ou NaN → une seule ligne avec NaN
    if not isinstance(equip_list, list) or len(equip_list) == 0:
        rows.append({
            "Nom affiché": nom,
            "Étiquettes": np.nan
        })

    # Cas 2 : liste non vide → une ligne par équipement
    else:
        for equip in equip_list:
            rows.append({
                "Nom affiché": nom,
                "Étiquettes": equip
            })

# Nouveau DataFrame avec seulement les 2 colonnes voulues
df_Etiquettes_fusion = pd.DataFrame(rows)

df_Etiquettes_fusion['Nom affiché'] = df_Etiquettes_fusion['Nom affiché'].str.strip()
df_Etiquettes_fusion['Étiquettes'] = df_Etiquettes_fusion['Étiquettes'].str.strip()

In [ ]:
df_Etiquettes_fusion

In [ ]:
df_Etiquettes_fusion = df_Etiquettes_fusion.dropna(subset=['Étiquettes'])

In [ ]:
output_file = "df_Etiquettes_fusion.xlsx"
df_Etiquettes_fusion.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)